In [8]:
from lib.hw_build import HwBuildHelper
from lib.sw_build import SwBuildHelper

In [9]:
board_build_tcl = "./kv260/board_build.tcl"
constraint_xdc  = "./kv260/constraint.xdc"

In [10]:
# ── Streamer definitions ────────────────────────────────────────────────────
# Index 0 is always the DMA pass-through streamer.
# Each entry: load_width / store_width in bytes (must be power of two),
#             actual_width in bits (<= load/store_width * 8),
#             amount_row: BRAM depth.

dfx_streamers_meta = [
    # streamer 0
    {"bank_req_per_set"   : 1,      # amount bank used in each set (cannot lower or higher )
     "amount_set"         : 1,      # amount of replication of bank set
     "amount_row_per_set" : 4096,   # row per set
     "width_per_bank"     : 32//8       # width in byte
     },
    ################### store from region 0
    # streamer 1
    {"bank_req_per_set"   : 2,
     "amount_set"         : 2,   # 16 bank in total 2*8
     "amount_row_per_set" : 4096,
     "width_per_bank"     : 8    # uram each bank have
     },
    # steramer 2
    {"bank_req_per_set"   : 4,
     "amount_set"         : 5,   # 16 bank in total 4*4
     "amount_row_per_set" : 4096,
     "width_per_bank"     : 8    # uram each bank have
     },
    ################### store from region 1
    # steramer 3
    {"bank_req_per_set"   : 2,
     "amount_set"         : 20,   # 16 bank in total 4*4
     "amount_row_per_set" : 4096,
     "width_per_bank"     : 8    # uram each bank have
     },

]

def build_dfx_streamers(meta_list):
    """Build dfx_streamers from dfx_streamers_meta.

    For each entry:
        load_width   = bank_req_per_set * width_per_bank   (bytes)
        store_width  = load_width
        actual_width = load_width * 8                      (bits)
        amount_row   = amount_set * amount_row_per_set
    """
    streamers = []
    for m in meta_list:
        load_width   = m["bank_req_per_set"] * m["width_per_bank"]
        store_width  = load_width
        actual_width = load_width * 8
        amount_row   = m["amount_set"] * m["amount_row_per_set"]
        streamers.append({
            "load_width"  : load_width,
            "store_width" : store_width,
            "actual_width": actual_width,
            "amount_row"  : amount_row,
        })
    return streamers

dfx_streamers = build_dfx_streamers(dfx_streamers_meta)
for i, s in enumerate(dfx_streamers):
    print(f"  streamer {i}: {s}")

# ── Region definitions ──────────────────────────────────────────────────────
# Single reconfigurable region wired to all 4 streamers (s0=DMA, s1, s2, s3).
# load_streamers / store_streamers: list of streamer indices connected to this region.
dfx_regions = [
    {"load_streamers": [0, 1, 2, 3]   , "store_streamers": [0, 1, 2, 3] },
]

# ── Reconfigurable module (RM) schematics ───────────────────────────────────
# 2-D list: rm_schemetics[region_idx][rm_idx]
# load_io_map / store_io_map: list of (streamer_index, kernel_port_index) pairs.
# kernel_port_index is fixed to 0 for all entries.
#
# Region 0: load from [s0, s1, s2, s3], store to [s0, s1, s2, s3], 2 RMs.
# Both RMs use empty I/O maps (test loopback).
rm_schemetics = [
    [  # region 0: load_streamers=[s0,s1,s2,s3], store_streamers=[s0,s1,s2,s3]
        {"load_io_map": [], "store_io_map": []},  # rm_0
        {"load_io_map": [], "store_io_map": []},  # rm_1
    ],
]

# ── Instantiate HwBuildHelper ───────────────────────────────────────────────
hw_builder = HwBuildHelper(
    build_folder_path="./build_prj",
    dfx_root_path="../..",
    board="custom",
    board_build_tcl=board_build_tcl,
    constraint_xdc=constraint_xdc,
    user_repo_path="",
    user_rm_build_tcl_path="",
    req_gen_ip=1,
    num_core=4,
    clk_frq=99999001,          # Hz
    rm_index_width=2,           # 1 << rm_index_width = max bank-1 slots; 2 RMs < 4
    dfx_streamers=dfx_streamers,
    dfx_regions=dfx_regions,
    rm_schemetics=rm_schemetics,
    test_mode=1,
    vivado_path="/tools/Xilinx/Vivado/2023.2/bin/vivado",
    export_folder_path="./export"
)

  streamer 0: {'load_width': 4, 'store_width': 4, 'actual_width': 32, 'amount_row': 4096}
  streamer 1: {'load_width': 16, 'store_width': 16, 'actual_width': 128, 'amount_row': 8192}
  streamer 2: {'load_width': 32, 'store_width': 32, 'actual_width': 256, 'amount_row': 20480}
  streamer 3: {'load_width': 16, 'store_width': 16, 'actual_width': 128, 'amount_row': 81920}


In [11]:
#hw_builder.run_build()


In [12]:
hw_builder.package_export_files()

In [13]:
sw_builder = SwBuildHelper(export_folder_path="./export", num_pr_region = 1, rm_index_width = 2, num_streamer = 4)

In [14]:
sw_builder.package_export_file()